In [0]:
dbutils.widgets.text("p_batch_id", ' ')
v_batch_id =  dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.Environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"

In [0]:
from pyspark.sql import functions as F

### Step 1 - Read bronze circuits table


In [0]:
circuits_df = spark.table(bronze_table).filter(F.col("batch_id") == v_batch_id)

In [0]:
display(circuits_df)

### Step 2 - keep required cols

In [0]:
circuits_selected_df = circuits_df.select (
    F.col("circuitId"),
    F.col('circuitName'),
    F.col('lat'),
    F.col('long'),
    F.col('locality'),
    F.col('country'),
    F.col('ingestion_timestamp'),
    F.col('source_file'),
    F.col('batch_id')
)

### step 3 & 4 - Standarise column names 

In [0]:
circuits_renamed_df = (
    circuits_selected_df
        .withColumnsRenamed({
            'circuitId':'circuit_id',
            'circuitName':'circuit_name',
            'lat':'latitude',
            'long':'longitude'
        })
)

### step 5 - filter the cols

In [0]:
circuits_valid_df = circuits_renamed_df.filter(
    F.col('circuit_id').isNotNull()
)

In [0]:
display(circuits_valid_df)

### step 6 - Remove the duplicate

In [0]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(['circuit_id'])

In [0]:
display(circuits_distinct_df)

### step 6 - Transform col values to title case

In [0]:
circuits_final_df = (
    circuits_distinct_df
        .withColumn('circuit_name', F.initcap(F.col('circuit_name')))
        .withColumn('locality', F.initcap(F.col('locality')))
)

In [0]:
display(circuits_final_df)

### step 8 - write table to silver layer

In [0]:
write_to_silver(
    input_df=circuits_final_df,
    target_table=silver_table,
    merge_condition="t.circuit_id = s.circuit_id",
    columns_to_update=[
        "circuit_name",
        "latitude",
        "longitude",
        "locality",
        "country",
        "ingestion_timestamp",
        "source_file",
        "batch_id"        
    ]
)

In [0]:
display(spark.table(silver_table))